# Pipeline Project

You will be using the provided data to create a machine learning model pipeline.

You must handle the data appropriately in your pipeline to predict whether an
item is recommended by a customer based on their review.
Note the data includes numerical, categorical, and text data.

You should ensure you properly train and evaluate your model.

## The Data

The dataset has been anonymized and cleaned of missing values.

There are 8 features for to use to predict whether a customer recommends or does
not recommend a product.
The `Recommended IND` column gives whether a customer recommends the product
where `1` is recommended and a `0` is not recommended.
This is your model's target/

The features can be summarized as the following:

- **Clothing ID**: Integer Categorical variable that refers to the specific piece being reviewed.
- **Age**: Positive Integer variable of the reviewers age.
- **Title**: String variable for the title of the review.
- **Review Text**: String variable for the review body.
- **Positive Feedback Count**: Positive Integer documenting the number of other customers who found this review positive.
- **Division Name**: Categorical name of the product high level division.
- **Department Name**: Categorical name of the product department name.
- **Class Name**: Categorical name of the product class name.

The target:
- **Recommended IND**: Binary variable stating where the customer recommends the product where 1 is recommended, 0 is not recommended.

## Load Data

In [1]:
import pandas as pd

# Load data
df = pd.read_csv(
    '../data/raw/reviews.csv',
)

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 18442 entries, 0 to 18441
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Clothing ID              18442 non-null  int64
 1   Age                      18442 non-null  int64
 2   Title                    18442 non-null  str  
 3   Review Text              18442 non-null  str  
 4   Positive Feedback Count  18442 non-null  int64
 5   Division Name            18442 non-null  str  
 6   Department Name          18442 non-null  str  
 7   Class Name               18442 non-null  str  
 8   Recommended IND          18442 non-null  int64
dtypes: int64(4), str(5)
memory usage: 1.3 MB


,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name,Recommended IND
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,0,General,Dresses,Dresses,0
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",0,General Petite,Bottoms,Pants,1
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,6,General,Tops,Blouses,1
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",4,General,Dresses,Dresses,0
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,1,General Petite,Tops,Knits,1


## Preparing features (`X`) & target (`y`)

In [2]:
data = df

# separate features from labels
X = data.drop('Recommended IND', axis=1)
y = data['Recommended IND'].copy()

print('Labels:', y.unique())
print('Features:')
display(X.head())

Labels: [0 1]
Features:


,Clothing ID,Age,Title,Review Text,Positive Feedback Count,Division Name,Department Name,Class Name
0,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,0,General,Dresses,Dresses
1,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",0,General Petite,Bottoms,Pants
2,847,47,Flattering shirt,This shirt is very flattering to all due to th...,6,General,Tops,Blouses
3,1080,49,Not for the very petite,"I love tracy reese dresses, but this one is no...",4,General,Dresses,Dresses
4,858,39,Cagrcoal shimmer fun,I aded this in my basket at hte last mintue to...,1,General Petite,Tops,Knits


In [3]:
# Split data into train and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    shuffle=True,
    random_state=27,
)

# Your Work

## Data Exploration

In [ ]:
# Target distribution
print("=== Target: Recommended IND ===")
counts = y.value_counts()
pcts = y.value_counts(normalize=True) * 100
for val in sorted(counts.index):
    label = "Recommended" if val == 1 else "Not Recommended"
    print(f"  {val} ({label}): {counts[val]:,}  ({pcts[val]:.1f}%)")
print(f"\nTotal rows: {len(y):,}")

In [ ]:
# Numerical features
num_features = ['Age', 'Positive Feedback Count']
print("=== Numerical Features ===")
display(X[num_features].describe().round(1))

In [ ]:
# Categorical features
cat_features = ['Division Name', 'Department Name', 'Class Name']
print("=== Categorical Features ===")
for col in cat_features:
    print(f"\n{col}:")
    print(X[col].value_counts().to_string())

In [ ]:
# Text features
text_features = ['Title', 'Review Text']
print("=== Text Features ===")
for col in text_features:
    lengths = X[col].str.len()
    words = X[col].str.split().str.len()
    print(f"\n{col}:")
    print(f"  Avg characters: {lengths.mean():.0f}  |  Max: {lengths.max()}")
    print(f"  Avg words:      {words.mean():.1f}  |  Max: {words.max()}")

## Building Pipeline

In [4]:
import numpy as np
import spacy
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, TransformerMixin

nlp = spacy.load('en_core_web_sm')

In [5]:
num_features = ['Age', 'Positive Feedback Count']
cat_features = ['Division Name', 'Department Name', 'Class Name']
review_text_feature = 'Review Text'
title_feature = 'Title'

In [6]:
class SpacyLemmatizer(BaseEstimator, TransformerMixin):
    def __init__(self, nlp):
        self.nlp = nlp

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [
            ' '.join([token.lemma_ for token in self.nlp(text)])
            if isinstance(text, str) else ''
            for text in X
        ]

In [7]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

review_text_pipeline = Pipeline([
    ('reshaper', FunctionTransformer(np.ravel)),
    ('lemmatizer', SpacyLemmatizer(nlp=nlp)),
    ('tfidf', TfidfVectorizer(stop_words='english')),
])

title_pipeline = Pipeline([
    ('reshaper', FunctionTransformer(np.ravel)),
    ('tfidf', TfidfVectorizer(stop_words='english')),
])

In [8]:
preprocessor = ColumnTransformer(transformers=[
    ('num',         num_pipeline,         num_features),
    ('cat',         cat_pipeline,         cat_features),
    ('review_text', review_text_pipeline, [review_text_feature]),
    ('title',       title_pipeline,       [title_feature]),
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=27)),
])

print("Pipeline built successfully!")
print(pipeline)

Pipeline built successfully!
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age',
                                                   'Positive Feedback Count']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unkno

## Training Pipeline

In [9]:
pipeline.fit(X_train, y_train)
print("Training complete!")

Training complete!


## Fine-Tuning Pipeline

In [10]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [None, 10, 20],
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions,
    n_iter=6,
    cv=5,
    scoring="accuracy",
    random_state=27,
    verbose=1,
)

search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print("Best CV score:", round(search.best_score_, 4))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'classifier__n_estimators': 100, 'classifier__max_depth': None}
Best CV score: 0.8588


# Evaluate

In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

best_pipeline = search.best_estimator_

y_pred = best_pipeline.predict(X_test)

print("=== Final Model Evaluation ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")

=== Final Model Evaluation ===
Accuracy:  0.8596
Precision: 0.8612
Recall:    0.9888
